# Optuna — Automated Hyperparameter Optimization

## What is Optuna?
**Optuna** is an automatic hyperparameter optimization framework — it finds the best hyperparameters for your ML model by running many trials intelligently.

### The Problem It Solves
Manual tuning or Grid Search wastes time:
- **Grid Search**: tries every combination → O(n^k) trials, most wasted
- **Random Search**: random combinations → better, but still blind
- **Optuna (Bayesian)**: learns from previous trials → focuses on promising regions

### Core Concepts
| Concept | Meaning |
|---------|---------|
| **Trial** | One run of the objective function with one set of hyperparameters |
| **Study** | A collection of trials — the full optimization run |
| **Objective function** | The function Optuna calls each trial; returns a score to minimize/maximize |
| **Sampler** | The strategy for choosing the next hyperparameter values |
| **Pruner** | Stops unpromising trials early to save time |

### What This Notebook Covers
1. Basic Optuna setup — TPE sampler on Random Forest
2. Samplers: TPE vs Random vs Grid
3. Visualizations — optimization history, parallel coordinates, importance
4. Multi-model search — SVM / RF / Gradient Boosting in one study
5. XGBoost with pruning — stop bad trials early


## Step 1: Install Optuna

In [38]:
!pip install optuna


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Load the Pima Indians Diabetes Dataset

**Dataset:** Pima Indians Diabetes (UCI Repository)
- 768 patients, 8 features, binary outcome (diabetic or not)

| Feature | Description |
|---------|-------------|
| Pregnancies | Number of pregnancies |
| Glucose | Plasma glucose concentration |
| BloodPressure | Diastolic blood pressure (mmHg) |
| SkinThickness | Triceps skinfold thickness (mm) |
| Insulin | 2-hour serum insulin |
| BMI | Body mass index |
| DiabetesPedigreeFunction | Genetic diabetes risk score |
| Age | Age in years |
| **Outcome** | **1 = diabetic, 0 = not diabetic** |


In [39]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Step 3: Clean Missing Values
Several columns use **0 as a placeholder for missing data** (e.g. Glucose=0 is medically impossible).

Fix: replace 0 → NaN → impute with column mean.


In [40]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


## Step 4: Train/Test Split + Feature Scaling
- 70% train, 30% test
- `StandardScaler` normalises features to mean=0, std=1 → helps most models converge faster


In [41]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


---
## Part 1: Basic Optuna — TPE Sampler

### What is TPE (Tree-structured Parzen Estimator)?
TPE is Optuna's **default and most powerful sampler**.

**How it works:**
1. First few trials → random exploration (like Random Search)
2. After gathering data → splits trials into "good" (top 25%) and "bad" (bottom 75%)
3. Fits two distributions: one over good hyperparameters, one over bad
4. Next trial = maximize `P(good) / P(bad)` → focuses on promising regions

**Result:** Far fewer trials needed compared to Grid or Random Search.

### The Objective Function
The objective function is called once per trial. It:
1. Receives a `trial` object
2. Uses `trial.suggest_*()` to pick hyperparameter values
3. Trains and evaluates a model
4. Returns the score (Optuna tries to maximize or minimize this)

```python
trial.suggest_int('n_estimators', 50, 200)   # integer in [50, 200]
trial.suggest_float('C', 0.01, 100, log=True) # float, log scale
trial.suggest_categorical('kernel', ['rbf', 'linear'])
```


In [42]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


### Create Study & Run 50 Trials
- `direction='maximize'` — we want to maximize accuracy
- `TPESampler` — Bayesian optimization strategy
- `n_trials=50` — run 50 evaluations

Each trial prints its result. Watch the best value climb as Optuna learns.


In [43]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-07-19 09:02:17,716] A new study created in memory with name: no-name-0f9184fd-e340-479e-a6c6-22e154948af1
[I 2026-07-19 09:02:18,428] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 130, 'max_depth': 11}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-07-19 09:02:19,187] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 136, 'max_depth': 10}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-07-19 09:02:19,570] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 71, 'max_depth': 14}. Best is trial 2 with value: 0.7690875232774674.
[I 2026-07-19 09:02:20,392] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 149, 'max_depth': 10}. Best is trial 2 with value: 0.7690875232774674.
[I 2026-07-19 09:02:21,112] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 160, 'max_depth': 12}. Best is trial 4 with value: 0.77094

### Best Trial Result

In [44]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 131, 'max_depth': 17}


### Train Final Model with Best Hyperparameters
Use the best hyperparameters found by Optuna to train on the full training set and evaluate on held-out test data.


In [45]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


---
## Part 2: Samplers in Optuna

A **sampler** controls how Optuna picks the next set of hyperparameters to try.

| Sampler | Strategy | Best for |
|---------|----------|---------|
| **TPESampler** (default) | Bayesian — learns from past trials | Most cases — efficient |
| **RandomSampler** | Pure random | Baseline comparison, parallel search |
| **GridSampler** | Exhaustive grid | Small, well-defined search spaces |
| **CmaEsSampler** | Evolution strategy | Continuous parameters, large spaces |


### 2a: Random Sampler
Picks hyperparameters uniformly at random — no learning between trials. Useful as a baseline.

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [47]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-07-19 09:02:52,044] A new study created in memory with name: no-name-45766cc4-3956-4fb6-bf5b-a3d9338045dd
[I 2026-07-19 09:02:52,401] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 60, 'max_depth': 9}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-07-19 09:02:52,802] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 86, 'max_depth': 5}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-07-19 09:02:53,344] Trial 2 finished with value: 0.7821229050279331 and parameters: {'n_estimators': 73, 'max_depth': 20}. Best is trial 2 with value: 0.7821229050279331.
[I 2026-07-19 09:02:54,828] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 168, 'max_depth': 20}. Best is trial 2 with value: 0.7821229050279331.
[I 2026-07-19 09:02:55,172] Trial 4 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 61, 'max_depth': 5}. Best is trial 2 with value: 0.78212290502

In [48]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279331
Best hyperparameters: {'n_estimators': 73, 'max_depth': 20}


In [49]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


### 2b: Grid Sampler
Exhaustively tries every combination in a predefined search space.

```
search_space = {n_estimators: [50,100,150,200], max_depth: [5,10,15,20]}
→ 4 × 4 = 16 total combinations — all will be tried
```

**When to use:** Small search space where you want guaranteed coverage.
**Avoid when:** Space is large — becomes exponentially expensive.


In [50]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [51]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-07-19 09:03:25,155] A new study created in memory with name: no-name-ddd8b544-c821-4c8a-abaf-5fca68c70fae
[I 2026-07-19 09:03:25,617] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-07-19 09:03:26,287] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-07-19 09:03:26,532] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-19 09:03:26,982] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-19 09:03:27,532] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [52]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [53]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


---
## Part 3: Optuna Visualizations

Optuna provides interactive Plotly charts to understand the optimization process.


## Optuna Visualizations

### Import Visualization Tools

In [54]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

### 1. Optimization History
Shows the **objective value** (accuracy) over each trial.
- Dots = individual trial scores
- Line = best score seen so far
- Look for: rapid improvement early → plateauing → good convergence


In [55]:
# 1. Optimization History
plot_optimization_history(study).show()

### 2. Parallel Coordinates Plot
Shows all trials as lines crossing vertical axes (one axis per hyperparameter).
- Lines that end at high accuracy = good hyperparameter combinations
- Cluster of good lines → stable region of the search space
- Useful for spotting which parameter ranges consistently perform well


In [56]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

### 3. Slice Plot
Shows how accuracy changes as **one hyperparameter varies** (others held constant).
- Each subplot = one hyperparameter
- Look for: clear peaks → optimal value range visible


In [57]:
# 3. Slice Plot
plot_slice(study).show()

### 4. Contour Plot
Shows the **interaction between two hyperparameters** as a heatmap.
- Bright regions = high accuracy combinations
- Useful for understanding parameter interactions (e.g. high n_estimators + low max_depth)


In [58]:
# 4. Contour Plot
plot_contour(study).show()

### 5. Hyperparameter Importance
Ranks which hyperparameters had the most impact on the objective score.
- Computed using **fANOVA** (functional ANOVA) over all trials
- High importance → tuning that parameter matters a lot
- Low importance → that parameter is not critical for this dataset/model


In [59]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

---
## Part 4: Optimizing Multiple ML Models in One Study

Instead of running separate studies for each model, put **all models in one objective function**.
Optuna simultaneously searches across:
- Which model to use (SVM, Random Forest, Gradient Boosting)
- The best hyperparameters for each model

### Key Trick: `suggest_categorical` for Model Selection
```python
classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])
```
Optuna treats model choice as just another hyperparameter — it learns which model tends to perform better.

### Hyperparameters per Model
| Model | Hyperparameters Tuned |
|-------|----------------------|
| SVM | C, kernel, gamma |
| Random Forest | n_estimators, max_depth, min_samples_split, min_samples_leaf, bootstrap |
| Gradient Boosting | n_estimators, learning_rate, max_depth, min_samples_split, min_samples_leaf |


## Optimizing Multiple ML Models

### Imports

In [60]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

### Multi-Model Objective Function

In [61]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

### Run 100 Trials Across All Models

In [62]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-07-19 09:03:36,586] A new study created in memory with name: no-name-216ab559-2dd1-4d62-8f67-1d9c0d8fafb0
[I 2026-07-19 09:03:36,624] Trial 0 finished with value: 0.7635009310986964 and parameters: {'classifier': 'SVM', 'C': 0.22210522924704215, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-19 09:03:38,155] Trial 1 finished with value: 0.7374301675977654 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 246, 'learning_rate': 0.2364019270414651, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-19 09:03:38,456] Trial 2 finished with value: 0.7672253258845437 and parameters: {'classifier': 'RandomForest', 'n_estimators': 64, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 6, 'bootstrap': False}. Best is trial 2 with value: 0.7672253258845437.
[I 2026-07-19 09:03:38,485] Trial 3 finished with value: 0.7579143389199254 and parame

### Best Trial — Winner Model & Parameters

In [63]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.13103309847801292, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


### All Trials as a DataFrame
Each row = one trial. Columns show hyperparameter values and the accuracy score.

In [64]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.763501,2026-07-19 09:03:36.588979,2026-07-19 09:03:36.624443,0 days 00:00:00.035464,0.222105,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.737430,2026-07-19 09:03:36.624443,2026-07-19 09:03:38.155879,0 days 00:00:01.531436,NaN,NaN,GradientBoosting,NaN,NaN,0.236402,5.0,7.0,9.0,246.0,COMPLETE
2,2,0.767225,2026-07-19 09:03:38.155879,2026-07-19 09:03:38.456124,0 days 00:00:00.300245,NaN,False,RandomForest,NaN,NaN,NaN,12.0,6.0,8.0,64.0,COMPLETE
3,3,0.757914,2026-07-19 09:03:38.456124,2026-07-19 09:03:38.485650,0 days 00:00:00.029526,4.488367,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.785847,2026-07-19 09:03:38.485650,2026-07-19 09:03:38.569252,0 days 00:00:00.083602,20.057993,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.789572,2026-07-19 09:04:06.885628,2026-07-19 09:04:06.911729,0 days 00:00:00.026101,0.120200,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.789572,2026-07-19 09:04:06.911729,2026-07-19 09:04:06.940191,0 days 00:00:00.028462,0.140759,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.785847,2026-07-19 09:04:06.941698,2026-07-19 09:04:06.961661,0 days 00:00:00.019963,0.272780,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.787709,2026-07-19 09:04:06.969013,2026-07-19 09:04:07.003114,0 days 00:00:00.034101,0.103268,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


### How Many Trials Did Each Model Get?
Optuna allocates more trials to models that seem promising.

In [65]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 80
GradientBoosting    10
RandomForest        10
Name: count, dtype: int64

### Average Accuracy by Model
Compare which model generally performed best across its trials.

In [66]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.744879
RandomForest        0.763315
SVM                 0.775256
Name: value, dtype: float64

### Optimization History — Multi-Model

In [67]:
# 1. Optimization History
plot_optimization_history(study).show()

### Slice Plot — Multi-Model
See how each hyperparameter affects accuracy within its model.

In [68]:
# 3. Slice Plot
plot_slice(study).show()

### Hyperparameter Importance — Multi-Model
Which hyperparameter (across all models) had the biggest impact?

In [69]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

---
## Part 5: XGBoost with Pruning (SuccessiveHalvingPruner)

### What is Pruning?
**Pruning** stops unpromising trials **early** — before they finish all boosting rounds.

```
Without pruning: every trial trains for 300 rounds
With pruning:    bad trials stopped at round 30, 60, etc. → 10× faster
```

### SuccessiveHalvingPruner
At each checkpoint:
- Keep only the top 50% of trials (by current performance)
- Discard the rest
- Allocate remaining budget to survivors

### XGBoost Hyperparameters Being Tuned
| Parameter | Role |
|-----------|------|
| `lambda` | L2 regularisation — prevents overfitting |
| `alpha` | L1 regularisation — sparsity |
| `eta` | Learning rate — step size per tree |
| `gamma` | Min loss reduction to split a node |
| `max_depth` | Tree depth — complexity |
| `min_child_weight` | Min samples in a leaf |
| `subsample` | Row sampling per tree |
| `colsample_bytree` | Feature sampling per tree |

### Dataset: Iris (3-class classification)
Uses `multi:softprob` objective — outputs probability for each of 3 classes.
`eval-mlogloss` = multi-class log loss used for pruning decisions.


In [70]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    from optuna_integration.xgboost import XGBoostPruningCallback
    pruning_callback = XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-07-19 09:04:07,659] A new study created in memory with name: no-name-51d35b29-6f15-4952-8dca-3c9cd3e6cf4f


[0]	train-mlogloss:1.06933	eval-mlogloss:1.06977
[1]	train-mlogloss:1.05370	eval-mlogloss:1.05279
[2]	train-mlogloss:1.03500	eval-mlogloss:1.03596
[3]	train-mlogloss:1.00336	eval-mlogloss:1.00203
[4]	train-mlogloss:0.98982	eval-mlogloss:0.98729
[5]	train-mlogloss:0.96853	eval-mlogloss:0.96462
[6]	train-mlogloss:0.93881	eval-mlogloss:0.93190
[7]	train-mlogloss:0.90858	eval-mlogloss:0.89902
[8]	train-mlogloss:0.89451	eval-mlogloss:0.88624
[9]	train-mlogloss:0.86499	eval-mlogloss:0.85537
[10]	train-mlogloss:0.84403	eval-mlogloss:0.83289
[11]	train-mlogloss:0.83331	eval-mlogloss:0.82360
[12]	train-mlogloss:0.81104	eval-mlogloss:0.79840
[13]	train-mlogloss:0.79287	eval-mlogloss:0.77965
[14]	train-mlogloss:0.76828	eval-mlogloss:0.75286
[15]	train-mlogloss:0.75487	eval-mlogloss:0.73919
[16]	train-mlogloss:0.74345	eval-mlogloss:0.72502
[17]	train-mlogloss:0.72531	eval-mlogloss:0.70670
[18]	train-mlogloss:0.70787	eval-mlogloss:0.68763
[19]	train-mlogloss:0.70332	eval-mlogloss:0.68373
[20]	train

[I 2026-07-19 09:04:08,900] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.1768922998962018, 'alpha': 0.030624106999035852, 'eta': 0.03152436890744786, 'gamma': 0.07440134925912152, 'max_depth': 4, 'min_child_weight': 7, 'subsample': 0.6218678294078834, 'colsample_bytree': 0.43426403773185257}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.06501	eval-mlogloss:1.06509
[1]	train-mlogloss:1.04809	eval-mlogloss:1.04703
[2]	train-mlogloss:1.02740	eval-mlogloss:1.02732
[3]	train-mlogloss:0.99178	eval-mlogloss:0.98905
[4]	train-mlogloss:0.97790	eval-mlogloss:0.97433
[5]	train-mlogloss:0.95556	eval-mlogloss:0.95276
[6]	train-mlogloss:0.92342	eval-mlogloss:0.91819
[7]	train-mlogloss:0.88937	eval-mlogloss:0.88111
[8]	train-mlogloss:0.87433	eval-mlogloss:0.86734
[9]	train-mlogloss:0.84285	eval-mlogloss:0.83441
[10]	train-mlogloss:0.81947	eval-mlogloss:0.80952
[11]	train-mlogloss:0.80898	eval-mlogloss:0.79880
[12]	train-mlogloss:0.78545	eval-mlogloss:0.77419
[13]	train-mlogloss:0.76602	eval-mlogloss:0.75407
[14]	train-mlogloss:0.73972	eval-mlogloss:0.72516
[15]	train-mlogloss:0.72495	eval-mlogloss:0.71011
[16]	train-mlogloss:0.71369	eval-mlogloss:0.69700
[17]	train-mlogloss:0.69529	eval-mlogloss:0.67886
[18]	train-mlogloss:0.67636	eval-mlogloss:0.65847
[19]	train-mlogloss:0.67155	eval-mlogloss:0.65376
[20]	train

[I 2026-07-19 09:04:11,144] Trial 1 finished with value: 1.0 and parameters: {'lambda': 2.2103251643439382e-07, 'alpha': 4.175424209513861e-08, 'eta': 0.03436484775445913, 'gamma': 0.07490304334488315, 'max_depth': 6, 'min_child_weight': 9, 'subsample': 0.8891669007431254, 'colsample_bytree': 0.4246211933340255}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.84294	eval-mlogloss:0.82770
[1]	train-mlogloss:0.75048	eval-mlogloss:0.73708


[I 2026-07-19 09:04:16,531] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:1.07098	eval-mlogloss:1.07168
[1]	train-mlogloss:1.05719	eval-mlogloss:1.05913


[I 2026-07-19 09:04:16,573] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 0.1768922998962018, 'alpha': 0.030624106999035852, 'eta': 0.03152436890744786, 'gamma': 0.07440134925912152, 'max_depth': 4, 'min_child_weight': 7, 'subsample': 0.6218678294078834, 'colsample_bytree': 0.43426403773185257}
Best accuracy: 1.0


### Install XGBoost Integration for Optuna

In [71]:
! pip install optuna-integration[xgboost]


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Plot Intermediate Values
Shows how each trial's `eval-mlogloss` evolves across boosting rounds.
- Lines that drop fast and stay low = good trials
- Lines that plateau high = correctly pruned early
- Pruned trials appear shorter — they were stopped before round 300


In [72]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()

---
## Summary

```
Optuna — Key Points:

  Core Workflow:
    1. Define objective(trial) → returns a scalar score
    2. study = optuna.create_study(direction='maximize')
    3. study.optimize(objective, n_trials=N)
    4. study.best_trial.params → use these to train final model

  Suggest API:
    trial.suggest_int('name', low, high)
    trial.suggest_float('name', low, high, log=True)
    trial.suggest_categorical('name', ['a', 'b', 'c'])

  Samplers:
    TPESampler   → Bayesian (default, recommended)
    RandomSampler → baseline / parallel jobs
    GridSampler  → small explicit grids

  Pruners (stop bad trials early):
    MedianPruner              → stop if below median at checkpoint
    SuccessiveHalvingPruner   → keep top 50% at each stage

  Visualizations:
    plot_optimization_history  → score over trials
    plot_parallel_coordinate   → all params vs score
    plot_param_importances     → which params matter most
    plot_intermediate_values   → per-round scores (pruning view)

  Multi-model trick:
    suggest_categorical('classifier', ['SVM','RF','GB'])
    → Optuna finds best model + best params in one study
```
